# Part 3: Topics

An entity graph shows how specific things are connected. **Topics** group related chunks so you can explore a broader area of the documents.

The hierarchy has two levels: broad topics at level 0 and narrower children at level 1. Each topic has a title and description to help you choose an area to explore. You'll use its **centroid**, a vector representing the group, to find similar source chunks.

**🎯 Mission**
- Check how much of the document set the topics cover.
- Browse the topic hierarchy.
- Pick a chunk and find its nearest neighbors in two ways: directly, and through its nearest topic.
- Query the sitemap with GQL, then use its inline-ontology knowledge source from a knowledge base.

You browsed these topics in the graph explorer's **Topics** tab. Now you'll query them with code.

## Step 1: Load your settings and connect to topics and PDF chunks

You'll use two indexes: `cocoarynth-topics` for the topic hierarchy, titles, descriptions, and centroid vectors, and `cocoarynth-chunks` for the PDF text and embeddings. Query the first to choose a topic, then search the second for related chunks.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True, raise_error_if_not_found=True)
load_dotenv(env_file, override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
API_VERSION = "2026-11-01-preview"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

In [ ]:
import json
import textwrap
from pathlib import Path

from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery

TOPICS_INDEX = "cocoarynth-topics"
SOURCES_INDEX = "cocoarynth-chunks"
TOPICS_KNOWLEDGE_SOURCE = f"{TOPICS_INDEX}-knowledge-source"

topics_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, TOPICS_INDEX, search_credential, api_version=API_VERSION
)
sources_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, SOURCES_INDEX, search_credential, api_version=API_VERSION
)
for index_name, client in ((TOPICS_INDEX, topics_client), (SOURCES_INDEX, sources_client)):
    print(f"{index_name}: {client.get_document_count()} documents")

## Step 2: Check topic coverage

The [lab metadata](../../data/graph-data/manifest.json) records the size and coverage of the topic hierarchy. Run the cell below to print these counts, and compare them with the tree in the explorer.

The topic groups cover 97 of the lab's 106 chunks. All 97 belong to the four broad topics; 61 also belong to narrower children. When you select a topic, the explorer searches the full source index with the topic's centroid to find similar chunks.

In [ ]:
metadata_file = Path(env_file).parent / "data" / "graph-data" / "manifest.json"
build_metadata = json.loads(metadata_file.read_text(encoding="utf-8"))["topics"]
assert build_metadata["schema_version"] == "topicgraph-release-v1"

for key in (
    "schema_version",
    "source_document_count",
    "clustered_source_document_count",
    "unassigned_source_document_count",
    "node_coverage",
    "label_stage",
    "assigned_chunks_by_level",
    "terminal_topic_count",
):
    print(f"{key:<34} {build_metadata[key]}")

## Step 3: Query the topic hierarchy

Read the hierarchy you browsed in the explorer. Each `Community` is a topic, and its facts describe the topic's properties and links. Use `predicate` to identify each kind of fact:

* `HasRootCommunity` connects the graph to its level-0 roots; `HasSubCommunity` connects parents to children.
* `prop:title` and `prop:description` use `object_string`; `prop:level`, `prop:proportion`, and `prop:parentProportion` use `object_double`.
* `HasCentroidVector` stores a leaf's `object_vector`.

A topic's count, printed next to its title, is its proportion multiplied by the number of clustered source chunks (97, not all 106), rounded.

Use a topic's `community:<n>` ID to look it up in later queries. A **terminal topic** has no children: it can be a narrower child or one of the two broad topics that stand alone.

In [ ]:
topic_facts = list(topics_client.search(
    search_text="*",
    select=["subject_type", "subject_id", "predicate", "object_id", "object_string", "object_double", "object_vector"],
    top=1000,
))
topics_by_id = {}
property_fields = {
    "prop:title": ("label", "object_string"),
    "prop:description": ("description", "object_string"),
    "prop:level": ("level", "object_double"),
    "prop:proportion": ("proportion", "object_double"),
    "HasCentroidVector": ("centroid", "object_vector"),
}
for fact in topic_facts:
    if fact["subject_type"] == "Community":
        topic = topics_by_id.setdefault(fact["subject_id"], {"topic_id": fact["subject_id"], "parent_id": None, "children": []})
        if fact["predicate"] in property_fields:
            name, field = property_fields[fact["predicate"]]
            topic[name] = fact[field]
for fact in topic_facts:
    if fact["predicate"] == "HasSubCommunity":
        topics_by_id[fact["subject_id"]]["children"].append(fact["object_id"])
        topics_by_id[fact["object_id"]]["parent_id"] = fact["subject_id"]


def leaf_topics(topic):
    return [leaf for child in topic["children"] for leaf in leaf_topics(topics_by_id[child])] if topic["children"] else [topic]


for topic in topics_by_id.values():
    topic["level"] = int(topic["level"])
    topic["member_count"] = round(topic["proportion"] * build_metadata["clustered_source_document_count"])
    topic["is_terminal"] = not topic["children"]
    if not topic["is_terminal"]:
        leaves = leaf_topics(topic)
        total = sum(leaf["proportion"] for leaf in leaves)
        topic["centroid"] = [
            sum(leaf["proportion"] / total * leaf["centroid"][i] for leaf in leaves)
            for i in range(build_metadata["dimension"])
        ]

for parent in sorted((t for t in topics_by_id.values() if t["level"] == 0), key=lambda t: t["topic_id"]):
    print(f"{parent['label']} ({parent['member_count']} chunks; terminal={parent['is_terminal']})")
    print(f"  {parent['description']}")
    for child_id in sorted(parent["children"]):
        child = topics_by_id[child_id]
        print(f"  └─ {child['label']} ({child['member_count']} chunks)")

## Step 4: Pick a chunk to start from

Find a chunk that mentions Madagascar bean lot MG26-L02 and reuse its stored embedding. This asks "what's near this chunk?" rather than embedding a new question, so no embedding model is called.

In [ ]:
start_chunk = list(sources_client.search(
    search_text='"MG26-L02"',
    search_fields=["chunk"],
    select=["chunk_id", "title", "blob_path", "chunk", "text_vector"],
    top=1,
))[0]
query_vector = start_chunk["text_vector"]
print(f"{start_chunk['title']} ({start_chunk['blob_path']})")
print(textwrap.indent(textwrap.shorten(start_chunk["chunk"], 400), "    "))

## Step 5: Find the nearest chunks directly

Run a vector search over all source chunks using the starting chunk's embedding. The first result is usually the starting chunk itself.

In [ ]:
direct = list(sources_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    select=["chunk_id", "title", "blob_path"],
    top=8,
))

for rank, result in enumerate(direct, 1):
    print(f"{rank}. {result['title']} ({result['blob_path']})")

## Step 6: Find the nearest topic, then search within it

Now route the search through a topic:

1. Find the leaf centroid closest to the starting chunk's embedding, filtering on `predicate eq 'HasCentroidVector'` and searching `object_vector`. Choose among all terminal topics, including the two broad roots at level 0.
2. Query `cocoarynth-chunks` for that centroid's nearest chunks, up to the topic's count (capped at 1,000). These approximate the topic's core.
3. Run the original chunk-vector query again, filtered to those candidates.

This narrows the candidate set but adds two vector queries and can miss relevant chunks.

In [ ]:
nearest_centroid = list(topics_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="object_vector", k_nearest_neighbors=1, exhaustive=True)],
    filter="predicate eq 'HasCentroidVector'",
    select=["subject_id"],
    top=1,
))[0]
nearest_topic = topics_by_id[nearest_centroid["subject_id"]]
member_count = min(nearest_topic["member_count"], 1000)
member_ids = [
    member["chunk_id"]
    for member in sources_client.search(
        vector_queries=[VectorizedQuery(vector=nearest_topic["centroid"], fields="text_vector", k_nearest_neighbors=member_count, exhaustive=True)],
        select=["chunk_id"],
        top=member_count,
    )
]
routed = list(sources_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    filter=f"search.in(chunk_id, '{','.join(member_ids)}', ',')",
    select=["chunk_id", "title", "blob_path"],
    top=8,
))

print(f"Nearest topic: {nearest_topic['label']} (level {nearest_topic['level']}; {len(member_ids)} nearest chunks)")
print(f"Starting chunk is among these candidates: {start_chunk['chunk_id'] in member_ids}\n")
for rank, result in enumerate(routed, 1):
    print(f"{rank}. {result['title']} ({result['blob_path']})")

## Step 7: Compare the two approaches

The direct search ran one vector query. Topic routing ran three: nearest leaf centroid, nearest source chunks for that centroid, and the original query within those candidates.

A chunk that appears in only one list wasn't returned by the other search. It's still in the index.

In [ ]:
direct_ids = {result["chunk_id"] for result in direct}
routed_ids = {result["chunk_id"] for result in routed}

print(f"Direct search:        {len(direct)} chunks (searched all chunks)")
print(f"Topic-routed search:  {len(routed)} chunks (searched {len(member_ids)} centroid-nearest candidates)")
print(f"\nIn both lists:        {len(direct_ids & routed_ids)}")
print(f"Only in direct list:  {len(direct_ids - routed_ids)}")
print(f"Only in topic list:   {len(routed_ids - direct_ids)}")

## Step 8: Query the sitemap with GQL

The graph profile exposes `CommunityGraph → HasRootCommunity → Community`, `Community → HasSubCommunity → Community`, and `Community → HasCentroidVector → Vector`.

Use GQL to find a broad topic by its `community:<n>` ID, follow it to its children, and read their descriptions. Then search the source index with each child's centroid to find example PDFs. Compare the topic names with the content of those documents.

In [ ]:
parent = next(topic for topic in topics_by_id.values() if not topic["is_terminal"])
PARENT_TOPIC_ID = parent["topic_id"]
SITEMAP_GQL = f"""
MATCH (parent:`Community`)-[:`HasSubCommunity`]->(topic:`Community`)
WHERE parent.id = '{PARENT_TOPIC_ID}'
RETURN parent.title AS parent, topic.id AS topic_id, topic.title AS topic, topic.description AS description
LIMIT 50
"""
sitemap_result = topics_client.graph(
    gql=SITEMAP_GQL,
    include_statistics=True,
)
for row in sitemap_result.value:
    print(f"{row['parent']} > {row['topic']}: {row['description']}")
    child = topics_by_id[row["topic_id"]]
    examples = sources_client.search(
        vector_queries=[VectorizedQuery(vector=child["centroid"], fields="text_vector", k_nearest_neighbors=min(child["member_count"], 1000), exhaustive=True)],
        select=["title", "blob_path"], top=2,
    )
    for example in examples:
        print(f"  Centroid-nearest PDF example (separate source search): {example['blob_path']}")

## Step 9: Use the graph-enabled topic knowledge source

`cocoarynth-topics-knowledge-source` connects a knowledge base to the topic index. Its `graphOntology` contains the inline JSON-LD serialization of the [sitemap OWL](../../data/graph-data/cocoarynth-sitemap.owl), which describes the topic types and relationships used by graph search.

![Diagram: cocoarynth-topics-knowledge-base uses cocoarynth-topics-knowledge-source to search CommunityGraph and Community facts with the inline sitemap ontology. Leaf centroids support vector searches over cocoarynth-chunks.](../../img/diagrams/notebook-03.svg)

Create a knowledge base over this source and ask it, in natural language, to summarize the same broad topic's children. Your handwritten query identified the parent by its ID; this request names the parent by its exact title and asks for its direct subtopics through `HasSubCommunity` relationships. Inspect whether a `graphSearch` step appears below the answer, then compare the summary with the titles and descriptions returned by your handwritten query.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters, KnowledgeBase, KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model, KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage, KnowledgeBaseMessageTextContent, KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort, KnowledgeRetrievalOutputMode,
    SearchIndexKnowledgeSourceParams,
)

index_client = SearchIndexClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, search_credential, api_version=API_VERSION
)
topic_source = index_client.get_knowledge_source(TOPICS_KNOWLEDGE_SOURCE).as_dict()
topic_ontology = topic_source["searchIndexParameters"].get("graphOntology", {})
print(f"Inline sitemap ontology: {topic_ontology['name']}")

TOPICS_KB_NAME = "cocoarynth-topics-knowledge-base"
index_client.create_or_update_knowledge_base(KnowledgeBase(
    name=TOPICS_KB_NAME,
    description="Navigate the canonical Cocoarynth topic hierarchy using its inline OWL schema",
    knowledge_sources=[KnowledgeSourceReference(name=TOPICS_KNOWLEDGE_SOURCE)],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
            resource_url=AZURE_OPENAI_ENDPOINT, deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
            model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME, api_key=AZURE_OPENAI_KEY,
        )),
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Use returned topic titles and descriptions only and cite reference IDs. Do not invent PDF membership or business facts.",
))
topics_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=TOPICS_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
topic_result = topics_kb_client.retrieve(KnowledgeBaseRetrievalRequest(
    messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(
        text="Use the sitemap graph's HasSubCommunity relationships to list direct subtopics "
             f"of the Community whose title is exactly {parent['label']!r}. "
             "For each subtopic, return its title and summarize its description."
    )])],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    include_activity=True,
    knowledge_source_params=[SearchIndexKnowledgeSourceParams(
        knowledge_source_name=TOPICS_KNOWLEDGE_SOURCE, include_references=True,
        include_reference_source_data=True, always_query_source=True,
    )],
))
display(Markdown(topic_result.response[0].content[0].text))
topic_graph_steps = [
    step.as_dict() for step in topic_result.activity or []
    if step.as_dict().get("type") == "graphSearch"
]
for step in topic_graph_steps:
    print(json.dumps(step, indent=2))

## Choose the right approach for the question

| Need | Good starting point | Tradeoff |
| --- | --- | --- |
| Ground a direct question in original documents | Standard knowledge base | The model plans queries, picks sources, and writes a cited answer |
| Retrieve chunks quickly | Microsoft Retriever 1 (MR1) | Faster planning without a chat completion model, but it searches every source and returns chunks, not an answer |
| Follow explicit relationships across records | Entity graph and GQL | Inspectable joins and provenance, but depends on the quality of the stored relationships |
| Explore groups of related chunks | Topics | Similarity-based browsing, topic hierarchy, and sitemap GQL |

Think about the three questions from this lab: the organic seal, the farms behind batch EC26-0811, and the Madagascar bean lot. Which approach would you start with for each, and why?

## ✅ Mission complete

**What you did:**

* ✓ **Topic coverage**: Checked how many source chunks belong to the topic groups.
* ✓ **Topic hierarchy**: Browsed the parent and child topics with their titles and descriptions.
* ✓ **Topic-routed search**: Compared nearest-neighbor search across all chunks with search through the nearest topic.
* ✓ **Sitemap**: Queried a broad topic's children with GQL and retrieved their titles and descriptions through the inline-ontology knowledge source. The PDF examples came from separate centroid searches of the source index.

➡️ Return to lab instructions.